# HW5: Lensless Computational Imaging

Лучший итоговый метод: `Pre4+LeADMM5+Post4`. На test split DigiCam он достигает `PSNR = 17.2964`.

In [ ]:
from pathlib import Path
import os
import sys

if Path.cwd().name == "analysis":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

## Как заново запустить inference

Полный inference на test split долгий, поэтому он вынесен из ipynb. При необходимости команды ниже запускаются из корня репозитория.

```bash
python3 scripts/download_checkpoints.py --output-dir checkpoints

python3 inference.py model=admm100 \
  inferencer.save_path=admm100_test \
  inferencer.skip_model_load=true

python3 inference.py model=fista \
  inferencer.save_path=fista_test \
  inferencer.skip_model_load=true

python3 inference.py model=leadmm20 \
  inferencer.save_path=leadmm20_test \
  inferencer.from_pretrained=checkpoints/leadmm20.pth \
  inferencer.skip_model_load=false

python3 inference.py model=modular_pre_only \
  inferencer.save_path=modular_pre_only_test \
  inferencer.from_pretrained=checkpoints/modular_pre_only.pth \
  inferencer.skip_model_load=false

python3 inference.py model=modular_post_only \
  inferencer.save_path=modular_post_only_test \
  inferencer.from_pretrained=checkpoints/modular_post_only.pth \
  inferencer.skip_model_load=false

python3 inference.py model=modular_pre_post \
  inferencer.save_path=modular_pre_post_test \
  inferencer.from_pretrained=checkpoints/modular_pre_post.pth \
  inferencer.skip_model_load=false
```

Метрики на DigiCam test split пересчитываются этими же inference-командами. Для своих данных с `lensed/` можно отдельно пересчитать метрики по PNG:

```bash
python3 scripts/calculate_metrics.py \
  --gt-dir /path/to/data_root/lensed \
  --recon-dir data/saved/custom_reconstructions/test
```

## Воспроизводимость

| Параметр | Значение |
|---|---|
| Dataset | `bezzam/DigiCam-Mirflickr-MultiMask-10K` |
| Train split | `train`, 5% из которого взято для валидации и выбора лучшей модели |
| Test split | `test`, 1500 изображений |
| Device | RTX 3080 CUDA |
| Seed | `42`, через `src/utils/init_utils.py` |

## Реализованные методы

`ADMM100`, `LeADMM20`, `Pre4+LeADMM5+Post4`, `Pre8+LeADMM5`, `LeADMM5+Post8`, `FISTA100`

## Validation Metrics

| Run | Best Val Epoch | Best Val PSNR | Best Val SSIM | Best Val MSE | Best Val LPIPS |
|---|---:|---:|---:|---:|---:|
| LeADMM20 | 11 | 13.7032 | 0.3958 | 0.049102 | 0.7394 |
| Pre4+LeADMM5+Post4 | 25 | 17.3113 | 0.4681 | 0.024705 | 0.5661 |
| Pre8+LeADMM5 | 23 | 14.8487 | 0.3176 | 0.038602 | 0.6970 |
| LeADMM5+Post8 | 22 | 16.6775 | 0.4541 | 0.027886 | 0.5874 |

## Test Metrics


| Method | Test PSNR | Test SSIM | Test MSE | Test LPIPS |
|---|---:|---:|---:|---:|
| Pre4+LeADMM5+Post4 | 17.2964 | 0.4599 | 0.023356 | 0.5655 |
| LeADMM5+Post8 | 16.7523 | 0.4468 | 0.026163 | 0.5842 |
| Pre8+LeADMM5 | 14.9538 | 0.3162 | 0.036932 | 0.6984 |
| LeADMM20 | 13.7775 | 0.3866 | 0.047398 | 0.7380 |
| ADMM100 | 11.9928 | 0.3493 | 0.076489 | 0.7808 |
| FISTA100 | 11.9588 | 0.2683 | 0.077316 | 0.7651 |

Для начала отметим что в порядке убывания лучшей эпохи по качеству PSNR на валидации модели идут так: `Pre4+LeADMM5+Post4`, `LeADMM5+Post8`, `Pre8+LeADMM5`, `LeADMM20`. На тесте ситуация по PSNR такая же, но со значением еще ниже добавляется `ADMM100` м потом еще ниже `FISTA100`.  По остальным метрикам рейтинг такой же за исклбчениями: 1) по SSIM `Pre8+LeADMM5` оказывается между `ADMM100` и `FISTA100`; 2) по LPIPS `ADMM100` оказывается худшим. В целом самое заметное, что `Pre4+LeADMM5+Post4` и `LeADMM5+Post8` заметно лучше всех осталльных, и разрыв по всем метрикам между ними ощутимо меньще чем разрыв до следующих моделей. При этом между нимим по всем метрикам `Pre4+LeADMM5+Post4` лучше `LeADMM5+Post8`, и как мне кажется лучше реально, а не на погрешность. Так же получается, что необучаемые `ADMM100` и `FISTA100` хуже всех остальных по всем метрикам (кроме SSIM где `Pre8+LeADMM5` становистя между ними), при этом также разрыв между ними по 3/4 метрик оказывается сильно меньше разрыва до следущих в топе (но уже выше) моделей. Тут уже мне не хочется говорить, что `FISTA100` однозначно хуже `ADMM100` потому что и разрыв реально меньше, и на LPIPS ситуация вообще обратная. 

## Качественный анализ

In [ ]:
from analysis import (
    load_report_samples,
    show_all_methods_comparison,
    show_selected_sample_psnr,
)

samples = load_report_samples()

### Сравнение всех методов на четырех примерах

Сами примеры я выбрал на рандом, но зафиксировал так как дальше отчет очевидно опирается именно на конкретные картинки.

In [ ]:
show_all_methods_comparison(samples)

Если смотреть только на качество картинок без метрик, то тут основная картинка как мне кажется такая же, за исключением того что я бы поставил все таки `ADMM100` скорее ниже чем `FISTA100`. Для начала сравнивая `Pre4+LeADMM5+Post4` и `LeADMM5+Post8`хочу сказать что тут уже как по мне разрыв в качестве совсем небольшой и наверное даже не везде заметный: на первой и 3 картинке как мне кажется у `Pre4+LeADMM5+Post4` совсем незначиельно выше резкость, на 2 я не могу сказать какая картинка мне кажется более похожей на оригинал, а на 4 картинке я бы сказал что уже становится прям заметно что у `Pre4+LeADMM5+Post4` сильно меньше посторонних цветов по этому картинка выглядит сильно ближе к таргету. На оставшихся 4 моделях везде заметен ощутимый разноцветный шум, но на `Pre8+LeADMM5` и на `FISTA100` он как будто состоит в основном из мелкой разноцветной сетки, а на `LeADMM20` и `ADMM100` довольно плавно размазан. Если смотреть по последней картинке, то `FISTA100` и `ADMM100` однозначно не смогли получить хоть сколько-то приемлевый результат, `Pre8+LeADMM5` и `LeADMM20` в целом приемлемо, общая часть картинки понимаема, но плоховато по сравнению с `Pre4+LeADMM5+Post4` и `LeADMM5+Post8`. Между собой `Pre8+LeADMM5` наверное чуть лучше `LeADMM20` по этой картинке, из-за мелкой сетки шума картинка в общем получается как будто более правильная визуально чем с большими пятнами на `LeADMM20`. По первым трем картинкам такой разницы между оставшымися 4 моделями уже нет, но все таки я точно хочу выделить что `ADMM100` хуже всех из-за достаточно мыльной картинки на всех кадрах (как будто сделали даунсэмпл и потом обратно просто растянули), шумы наверное самые менее притичные в `LeADMM20` (но тут уже довольно субъективно, шум дело такое), но при этом `Pre8+LeADMM5` как будто давет везде самую четку картинку. 

## PSNR на выбранных визуальных примерах

In [ ]:
show_selected_sample_psnr(samples)

## Измерение скорости


| Method | Mean ms/image | Std ms |
|---|---:|---:|
| LeADMM5+Post8 | 37.503 | 0.077 |
| Pre8+LeADMM5 | 37.572 | 0.484 |
| Pre4+LeADMM5+Post4 | 50.076 | 0.077 |
| LeADMM20 | 90.288 | 0.046 |
| ADMM100 | 446.866 | 0.618 |
| FISTA100 | 2073.763 | 1.295 |

В общем подводя итоги по убыванию качества (субъективной совокупности количесттвенной оценки и качественной) я бы поставил модели в таком порядке: `Pre4+LeADMM5+Post4`, `LeADMM5+Post8`, `Pre8+LeADMM5`, `LeADMM20`, `FISTA100`, `ADMM100`, причем первые два места и последние довольно однозначно, а оставшиеся 3 модели я бы сказал что в зависимости от задачи и личных предпочтений могут быть по разному расположены.  
  
Если учитывать скорость обработки изображений то я бы делал в совокупности с качеством уже такой рейтинг: `LeADMM5+Post8`, `Pre4+LeADMM5+Post4`, `Pre8+LeADMM5`, `LeADMM20`, `ADMM100`, `FISTA100`, потому что как по мне разница в качестве у первых двух не такая большая как по времени, при этом они релаьно сильно качественнее остальных, `LeADMM20` оказываеьтся все таки заметно медленнее `Pre8+LeADMM5`, пости в 3 раза, и это уже серьезно, а `FISTA100` хоть по качеству я бы сказал она и превосходит `ADMM100`, но разница во времени коллосальная (в 4.5 раз больше). Ну и необучаемые методы в целом оказываются значительно хуже обучаемых и по времени и по качеству.